# EDA — Clasificación de Materiales Industriales

**Proyecto Final — Especialización Analítica y Ciencia de Datos**  
Universidad de Medellín

**Target:** `familia_cod` — 8 categorías de materiales industriales

## 0. Setup — Entorno Colab

In [ ]:
import os

REPO = 'proyecto_final_clasificacion_materiales'
REPO_URL = 'https://github.com/marianarada/proyecto_final_clasificacion_materiales.git'
REPO_PATH = f'/content/{REPO}'

# Clonar solo si no existe
if not os.path.exists(REPO_PATH):
    os.system(f'git clone {REPO_URL}')

os.chdir(REPO_PATH)
print('Working dir:', os.getcwd())

In [ ]:
# Instalar dependencias del proyecto
!pip install -e . -q

In [ ]:
# Mover archivos Excel al path esperado por load_dataset()
os.makedirs('data/raw', exist_ok=True)
os.makedirs('../reports/figures', exist_ok=True)
!cp Data/*.xlsx data/raw/ 2>/dev/null || true
print('Archivos en data/raw/:', os.listdir('data/raw'))

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config_loader import load_config
from src.data.load_data import load_dataset
from src.data.validate import validate_dataset

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
PALETTE = sns.color_palette('tab10', 8)

NOMBRES = {
    'LUB': 'Lubricantes', 'INS': 'Instrumentación', 'TUV': 'Tubería/Válvulas',
    'EPP': 'EPP', 'ELE': 'Eléctricos', 'QUI': 'Químicos',
    'HER': 'Herramientas', 'MEC': 'Mecánicos'
}

print('Imports OK')

In [ ]:
config = load_config()
df = load_dataset()
print(validate_dataset(df, config).summary())
print(f'Shape: {df.shape}')
df.head(3)

## 1. Distribución del Target

In [ ]:
counts = df['familia_cod'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].barh([NOMBRES[c] for c in counts.index], counts.values, color=PALETTE)
axes[0].bar_label(bars, fmt='%d', padding=4)
axes[0].set_title('Conteo por familia')
axes[0].invert_yaxis()

axes[1].pie(
    counts.values,
    labels=[NOMBRES[c] for c in counts.index],
    autopct='%1.1f%%', colors=PALETTE, startangle=140, pctdistance=0.82
)
axes[1].set_title('Proporción')

plt.suptitle('Distribución de clases', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/01_distribucion_clases.png', bbox_inches='tight')
plt.show()
print(f'Imbalance ratio: {counts.max()/counts.min():.2f}x  → dataset casi perfectamente balanceado')

## 2. Análisis de Texto

In [ ]:
df['n_breve'] = df['texto_breve'].str.split().str.len()
df['n_largo'] = df['texto_largo'].str.split().str.len()
df['n_total'] = df['texto_completo'].str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, t in zip(axes,
                      ['n_breve', 'n_largo', 'n_total'],
                      ['Texto breve', 'Texto largo', 'Texto completo']):
    ax.hist(df[col], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(df[col].mean(),   color='crimson', ls='--', label=f'Media: {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='orange',  ls='--', label=f'Mediana: {df[col].median():.1f}')
    ax.set_title(t)
    ax.set_xlabel('# palabras')
    ax.legend(fontsize=9)

plt.suptitle('Longitud de texto', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/02_longitud_texto.png', bbox_inches='tight')
plt.show()
print(df[['n_breve', 'n_largo', 'n_total']].describe().round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
order = df.groupby('familia_cod')['n_total'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='familia_cod', y='n_total', order=order, palette='tab10', ax=ax)
ax.set_xticklabels([NOMBRES[c] for c in order], rotation=20, ha='right')
ax.set_title('Longitud de texto completo por familia')
ax.set_xlabel('')
ax.set_ylabel('# palabras')
plt.tight_layout()
plt.savefig('../reports/figures/03_texto_por_familia.png', bbox_inches='tight')
plt.show()

## 3. Features Categóricas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_marcas = df['marca'].value_counts().head(10)
axes[0].barh(top_marcas.index[::-1], top_marcas.values[::-1], color='teal', alpha=0.8)
axes[0].set_title('Top 10 marcas')
pct_sm = (df['marca'] == 'sin_marca').mean() * 100
axes[0].set_xlabel(f'Frecuencia  |  Sin marca: {pct_sm:.1f}% del total')

top_um = df['um'].value_counts().head(8)
axes[1].bar(top_um.index, top_um.values, color='coral', alpha=0.8)
axes[1].set_title('Unidades de medida (UM)')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha='center', va='bottom', fontsize=9)

plt.suptitle('Features categóricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/04_features_categoricas.png', bbox_inches='tight')
plt.show()

## 4. Features Numéricas

In [ ]:
numeric_cols = ['precio_usd', 'filas_me2m', 'items_ariba']
num_labels   = ['Precio (USD)', 'Filas ME2M', 'Ítems Ariba']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, (col, lbl) in enumerate(zip(numeric_cols, num_labels)):
    # Fila 0: histograma en escala log
    ax = axes[0, i]
    vals = df[col][df[col] > 0]
    ax.hist(np.log1p(vals), bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f'{lbl} (log+1)')
    ax.set_xlabel('log(1+valor)')
    # Fila 1: boxplot por familia
    ax2 = axes[1, i]
    sns.boxplot(data=df, x='familia_cod', y=col,
                order=sorted(df['familia_cod'].unique()),
                palette='tab10', ax=ax2, showfliers=False)
    ax2.set_title(f'{lbl} por familia')
    ax2.set_xlabel('')
    ax2.tick_params(axis='x', rotation=45)

plt.suptitle('Features numéricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/05_features_numericas.png', bbox_inches='tight')
plt.show()
print(df[numeric_cols].describe().round(2))

## 5. Correlación

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, xticklabels=num_labels, yticklabels=num_labels,
            ax=ax, linewidths=0.5)
ax.set_title('Correlación features numéricas')
plt.tight_layout()
plt.savefig('../reports/figures/06_correlacion.png', bbox_inches='tight')
plt.show()

## 6. Conclusiones

In [ ]:
ratio = df['familia_cod'].value_counts().max() / df['familia_cod'].value_counts().min()

print('=' * 58)
print('RESUMEN EDA — Clasificación de Materiales')
print('=' * 58)
print(f"""
DATASET
  Filas: {len(df):,}  |  Columnas: {df.shape[1]}  |  Nulos: 0

TARGET (familia_cod)
  Clases: {df['familia_cod'].nunique()}  |  Imbalance ratio: {ratio:.2f}x
  Mayoritaria: {df['familia_cod'].value_counts().idxmax()} ({df['familia_cod'].value_counts().max()} items)
  Minoritaria:  {df['familia_cod'].value_counts().idxmin()} ({df['familia_cod'].value_counts().min()} items)

TEXTO
  Breve:  {df['n_breve'].mean():.1f} palabras avg  →  feature principal
  Largo:  {df['n_largo'].mean():.1f} palabras avg  →  concatenar con breve

MARCA
  Sin marca: {(df['marca']=='sin_marca').sum()} materiales ({(df['marca']=='sin_marca').mean():.1%})

DECISIONES DE MODELADO
  ✓ TF-IDF ngram(1,2) sobre texto_completo como feature principal
  ✓ Encoding one-hot para marca y um
  ✓ StandardScaler para numéricas (outliers presentes)
  ✓ class_weight='balanced' por precaución
  ✓ Baseline: LogisticRegression → RandomForest → XGBoost
""")